# Análise do Banco Sectra ERP — SQL Server
Exploração, consulta, exportação e visualização de dados via Python.


## 1. Bibliotecas


In [ ]:

import subprocess, sys

pkgs = ["pyodbc", "pandas", "sqlalchemy", "openpyxl", "rich", "python-dotenv",
        "matplotlib", "seaborn", "plotly", "tabulate"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=True)

import os, warnings
import pyodbc
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore")
load_dotenv()          # carrega o arquivo .env
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("✅ Bibliotecas carregadas com sucesso!")


## 2. Conexão com SQL Server

Preencha o arquivo `.env` na raiz do projeto com suas credenciais antes de executar.


In [ ]:

# ── Configuração ─────────────────────────────────────────────
SERVER   = os.getenv("DB_SERVER",   "SEU_SERVIDOR")   # ex: 192.168.1.10 ou HOST\INSTANCIA
PORT     = os.getenv("DB_PORT",     "1433")
DATABASE = os.getenv("DB_DATABASE", "NOME_DO_BANCO")
USERNAME = os.getenv("DB_USERNAME", "seu_usuario")
PASSWORD = os.getenv("DB_PASSWORD", "sua_senha")
DRIVER   = os.getenv("DB_DRIVER",   "ODBC Driver 17 for SQL Server")

# Drivers disponíveis na sua máquina:
print("Drivers ODBC instalados:")
for d in pyodbc.drivers():
    print(f"  • {d}")

# ── Monta a connection string ─────────────────────────────────
conn_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER},{PORT};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};PWD={PASSWORD};"
    "TrustServerCertificate=Yes;Readonly=Yes;"
)

params  = quote_plus(conn_str)
engine  = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", connect_args={"timeout": 10})

# ── Teste de conexão ─────────────────────────────────────────
with engine.connect() as conn:
    versao = conn.execute(text("SELECT @@VERSION")).scalar()

print(f"\n✅ Conectado!\n{versao[:100]}...")


## 3. Explorar Estrutura do Banco


In [ ]:

def sql(query, params=None):
    """Helper: executa uma query e retorna DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params or {})

# ── 3.1 Resumo de objetos no banco ───────────────────────────
resumo = sql("""
    SELECT type_desc AS tipo, COUNT(*) AS quantidade
    FROM sys.objects WHERE is_ms_shipped = 0
    GROUP BY type_desc ORDER BY quantidade DESC
""")
print("=== Objetos no Banco ===")
display(resumo)


In [ ]:

# ── 3.2 Todas as tabelas com contagem de linhas ──────────────
tabelas = sql("""
    SELECT
        t.TABLE_SCHEMA AS [schema],
        t.TABLE_NAME   AS tabela,
        t.TABLE_TYPE   AS tipo,
        p.rows         AS qtd_linhas
    FROM INFORMATION_SCHEMA.TABLES t
    LEFT JOIN sys.partitions p
           ON p.object_id = OBJECT_ID(t.TABLE_SCHEMA+'.'+t.TABLE_NAME)
          AND p.index_id IN (0,1)
    WHERE t.TABLE_TYPE = 'BASE TABLE'
    ORDER BY p.rows DESC, t.TABLE_NAME
""")
print(f"Total de tabelas: {len(tabelas)}")
display(tabelas.head(30))


In [ ]:

# ── 3.3 Descrever colunas de uma tabela ─────────────────────
TABELA = "NomeDaTabela"        # ← substitua pelo nome real
SCHEMA = "dbo"

colunas = sql("""
    SELECT COLUMN_NAME AS coluna, DATA_TYPE AS tipo,
           CHARACTER_MAXIMUM_LENGTH AS tamanho,
           IS_NULLABLE AS nulo, COLUMN_DEFAULT AS [default]
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = :t AND TABLE_SCHEMA = :s
    ORDER BY ORDINAL_POSITION
""", {"t": TABELA, "s": SCHEMA})
display(colunas)


In [ ]:

# ── 3.4 Relacionamentos (Chaves Estrangeiras) ────────────────
fks = sql("""
    SELECT fk.name AS fk_nome,
           tp.name AS tabela_origem,  cp.name AS coluna_origem,
           tr.name AS tabela_destino, cr.name AS coluna_destino
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    JOIN sys.tables  tp ON tp.object_id = fkc.parent_object_id
    JOIN sys.columns cp ON cp.object_id = fkc.parent_object_id   AND cp.column_id = fkc.parent_column_id
    JOIN sys.tables  tr ON tr.object_id = fkc.referenced_object_id
    JOIN sys.columns cr ON cr.object_id = fkc.referenced_object_id AND cr.column_id = fkc.referenced_column_id
    ORDER BY tabela_origem, fk_nome
""")
print(f"Relacionamentos encontrados: {len(fks)}")
display(fks.head(30))


In [ ]:

# ── 3.5 Buscar coluna por nome (útil para mapear o banco) ────
BUSCA = "data"      # ← parte do nome da coluna que você quer achar

resultado = sql("""
    SELECT TABLE_SCHEMA AS [schema], TABLE_NAME AS tabela,
           COLUMN_NAME AS coluna, DATA_TYPE AS tipo
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE COLUMN_NAME LIKE :nome
    ORDER BY TABLE_NAME, COLUMN_NAME
""", {"nome": f"%{BUSCA}%"})

print(f"Colunas contendo '{BUSCA}': {len(resultado)}")
display(resultado)


## 4. Consultar e Ler Dados


In [ ]:

# ── 4.1 Preview de uma tabela ────────────────────────────────
df = sql("SELECT TOP 20 * FROM [dbo].[NomeDaTabela]")   # ← substitua
print(f"Shape: {df.shape}")
display(df)


In [ ]:

# ── 4.2 Query com filtro, agrupamento e ordenação ────────────
# Adapte a query para o seu banco real

df_pedidos = sql("""
    SELECT
        YEAR(data_pedido)  AS ano,
        MONTH(data_pedido) AS mes,
        status,
        COUNT(*)           AS qtd_pedidos,
        SUM(valor_total)   AS faturamento
    FROM dbo.Pedidos
    WHERE data_pedido >= '2024-01-01'
    GROUP BY YEAR(data_pedido), MONTH(data_pedido), status
    ORDER BY ano, mes, status
""")
display(df_pedidos.head(20))


In [ ]:

# ── 4.3 JOIN entre tabelas ───────────────────────────────────
df_join = sql("""
    SELECT
        p.id              AS pedido_id,
        p.data_pedido,
        c.nome            AS cliente,
        c.cidade,
        p.valor_total,
        p.status
    FROM dbo.Pedidos p
    JOIN dbo.Clientes c ON c.id = p.cliente_id
    WHERE p.status = 'FATURADO'
      AND p.data_pedido >= '2025-01-01'
    ORDER BY p.data_pedido DESC
""")
print(f"Registros: {len(df_join):,}")
display(df_join.head(15))


## 5. Exportar Dados


In [ ]:

import os
from pathlib import Path
from datetime import datetime

OUTPUT = Path("exports")
OUTPUT.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

def ajustar_colunas(writer, aba, df):
    ws = writer.sheets[aba]
    for i, col in enumerate(df.columns, 1):
        largura = max(len(str(col)), df[col].astype(str).str.len().max() if len(df) > 0 else 0)
        ws.column_dimensions[ws.cell(1, i).column_letter].width = min(largura + 4, 60)

# ── 5.1 Exportar para CSV ────────────────────────────────────
caminho_csv = OUTPUT / f"dados_{ts}.csv"
df_pedidos.to_csv(caminho_csv, sep=";", index=False, encoding="utf-8-sig")
print(f"✅ CSV: {caminho_csv}")

# ── 5.2 Exportar para Excel (aba única) ─────────────────────
caminho_excel = OUTPUT / f"dados_{ts}.xlsx"
with pd.ExcelWriter(caminho_excel, engine="openpyxl") as writer:
    df_pedidos.to_excel(writer, sheet_name="Pedidos", index=False)
    ajustar_colunas(writer, "Pedidos", df_pedidos)
print(f"✅ Excel: {caminho_excel}")


In [ ]:

# ── 5.3 Excel com múltiplas abas ─────────────────────────────
abas = {
    "Pedidos por Mês": df_pedidos,
    # "Clientes":     df_clientes,   # ← adicione outros DataFrames
}

caminho_multi = OUTPUT / f"relatorio_geral_{ts}.xlsx"
with pd.ExcelWriter(caminho_multi, engine="openpyxl") as writer:
    for nome_aba, df_aba in abas.items():
        df_aba.to_excel(writer, sheet_name=nome_aba[:31], index=False)
        ajustar_colunas(writer, nome_aba[:31], df_aba)
        print(f"   ✔ '{nome_aba}': {len(df_aba):,} linhas")

print(f"\n✅ Excel multi-abas: {caminho_multi}")


## 6. Visualizações e Dashboards


In [ ]:

# ── 6.1 Faturamento mensal (matplotlib) ──────────────────────
# Adapte as colunas ao seu DataFrame real

df_fat = df_pedidos.groupby(["ano","mes"])["faturamento"].sum().reset_index()
df_fat["periodo"] = df_fat["ano"].astype(str) + "-" + df_fat["mes"].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(df_fat["periodo"], df_fat["faturamento"], color="#1f77b4")
ax.set_title("Faturamento Mensal", fontsize=14, fontweight="bold")
ax.set_xlabel("Período")
ax.set_ylabel("Faturamento (R$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

# ── 6.2 Pizza por status de pedido (plotly — interativo) ─────
df_status = df_pedidos.groupby("status")["qtd_pedidos"].sum().reset_index()

fig = px.pie(
    df_status,
    names="status",
    values="qtd_pedidos",
    title="Pedidos por Status",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textinfo="percent+label")
fig.show()


In [ ]:

# ── 6.3 Dashboard de KPIs (plotly) ───────────────────────────
fat_total   = df_pedidos["faturamento"].sum()
qtd_total   = df_pedidos["qtd_pedidos"].sum()
ticket_medio = fat_total / qtd_total if qtd_total else 0

fig = go.Figure()

kpis = [
    ("Faturamento Total",  f"R$ {fat_total:,.2f}",  "#2ecc71"),
    ("Total de Pedidos",   f"{qtd_total:,}",          "#3498db"),
    ("Ticket Médio",       f"R$ {ticket_medio:,.2f}", "#e67e22"),
]

for i, (titulo, valor, cor) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode="number",
        value=None,
        title={"text": f"<b>{titulo}</b><br><span style='font-size:1.5em;color:{cor}'>{valor}</span>"},
        domain={"column": i, "row": 0},
    ))

fig.update_layout(
    grid={"rows": 1, "columns": 3},
    height=200,
    margin=dict(t=60, b=10),
)
fig.show()


## 7. Gerar Relatório HTML Automático


In [ ]:

# ── 7.1 Relatório HTML completo ──────────────────────────────
# Gera um .html com tabelas e gráficos embutidos, que pode ser
# aberto no navegador ou enviado por e-mail.

import base64
from io import BytesIO

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# Gráfico: faturamento mensal
fig_bar, ax_bar = plt.subplots(figsize=(12, 4))
ax_bar.bar(df_fat["periodo"], df_fat["faturamento"], color="#1f77b4")
ax_bar.set_title("Faturamento Mensal")
ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
img_b64 = fig_to_base64(fig_bar)
plt.close()

html = f"""<!DOCTYPE html>
<html lang="pt-BR"><head>
<meta charset="UTF-8">
<title>Relatório ERP Sectra — {datetime.now().strftime('%d/%m/%Y')}</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 30px; color: #333; }}
  h1   {{ color: #1a5276; }}
  h2   {{ color: #2471a3; border-bottom: 2px solid #aed6f1; padding-bottom: 4px; }}
  .kpi {{ display: inline-block; margin: 10px 20px; padding: 16px 24px;
          background: #eaf4fb; border-radius: 8px; text-align: center; }}
  .kpi .val {{ font-size: 1.8em; font-weight: bold; color: #1a5276; }}
  table {{ border-collapse: collapse; width: 100%; font-size: .9em; }}
  th    {{ background: #2471a3; color: #fff; padding: 8px; }}
  td    {{ border: 1px solid #ddd; padding: 6px; }}
  tr:nth-child(even) {{ background: #f2f9ff; }}
</style></head><body>

<h1>Relatório ERP Sectra</h1>
<p>Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>

<h2>KPIs</h2>
<div class="kpi"><div class="val">R$ {fat_total:,.2f}</div>Faturamento Total</div>
<div class="kpi"><div class="val">{qtd_total:,}</div>Total de Pedidos</div>
<div class="kpi"><div class="val">R$ {ticket_medio:,.2f}</div>Ticket Médio</div>

<h2>Faturamento Mensal</h2>
<img src="data:image/png;base64,{img_b64}" width="900"/>

<h2>Dados Detalhados</h2>
{df_pedidos.to_html(index=False, border=0, classes="dataframe")}

</body></html>"""

caminho_html = OUTPUT / f"relatorio_{ts}.html"
caminho_html.write_text(html, encoding="utf-8")
print(f"✅ Relatório HTML: {caminho_html}")
